In [1]:
import pandas as pd

In [10]:
ICD_CODE = "C61"

In [20]:
MODEL = "GPT_OSS_MODEL_SIZE.SMALL_Low"
TRY = 0
scores = pd.read_csv(f"scores/scores_{MODEL}_{TRY}.csv").rename(columns={"Unnamed: 0": "icd10_category"})
scores = scores[["icd10_category", ICD_CODE]].rename(columns={ICD_CODE: "score"})

In [21]:
scores.head()

,icd10_category,score
0,A00,0.0
1,A01,0.0
2,A02,0.0
3,A03,0.0
4,A04,0.0


In [23]:
scores["score"].value_counts()

score
0.0    1992
0.5      52
Name: count, dtype: int64

In [ ]:
MODEL = "GPT_OSS_MODEL_SIZE.SMALL_Low"

TRY = 0
scores_mcq = pd.read_csv(f"scores_mcq/scores_{MODEL}_{ICD_CODE}_{TRY}.csv").drop("Unnamed: 0", axis=1, errors="ignore")

In [17]:
scores_mcq.head()

,icd10_category_1,icd10_category_2,score
0,C61,K76,0.0
1,C61,R18,0.0
2,C61,K74,0.0
3,C61,B19,0.0
4,C61,J44,0.0


In [5]:
scores_mcq['score'].value_counts()

score
0.0    1642
0.5      52
1.0       2
Name: count, dtype: int64

In [25]:
all_scores = pd.merge(
    scores,
    scores_mcq,
    how="inner",
    left_on="icd10_category",
    right_on="icd10_category_2",
    suffixes=("", "_mcq")
)

In [26]:
all_scores.shape

(1696, 5)

In [27]:
all_scores.head(2)

,icd10_category,score,icd10_category_1,icd10_category_2,score_mcq
0,A01,0.0,C61,A01,0.0
1,A02,0.0,C61,A02,0.0


In [28]:
all_scores["is_equal"] = (all_scores["score"] == all_scores["score_mcq"]).astype(int)

In [29]:
all_scores["is_equal"].sum()/len(all_scores)

0.9463443396226415

In [34]:
all_scores_no_zero = all_scores[(all_scores["score"] != 0) | (all_scores["score_mcq"] != 0)]
all_scores_no_zero["is_equal"].sum()/len(all_scores_no_zero)

0.07142857142857142

In [ ]:
import sklearn
import sklearn.metrics 

In [36]:
for hard_metric in ["accuracy", "f1", "precision", "recall"]:
    print(
        hard_metric, ":", 
        getattr(sklearn.metrics, hard_metric+"_score")(
            all_scores['score_mcq'] >= 0.5, 
            all_scores['score'] >= 0.5
        )
    )

for soft_metric in ["roc_auc", "average_precision"]:
    print(
        soft_metric, ":", 
        getattr(sklearn.metrics, soft_metric+"_score")(
            all_scores['score_mcq'] >= 0.5, 
            all_scores['score']
        )
    )

accuracy : 0.9469339622641509
f1 : 0.1509433962264151
precision : 0.15384615384615385
recall : 0.14814814814814814
roc_auc : 0.5606757793115893
average_precision : 0.04991466430145676
